# ShipsEar: CNN and ResNet18 Reproduction

This notebook implements the **CNN and ResNet18 baselines only**. STM/Transformer components are intentionally excluded.

Follow the course implementation guide: prepare 5-second clips, create Dataset A and Dataset B, extract STFT/FBank/MFCC features, train both baselines, and compare against the paper.

## 1. Environment and paths

In Colab, mount Drive and place the downloaded ShipsEar recordings plus the mapped metadata CSV in your Drive. Do not upload the raw dataset to GitHub.

In [ ]:
# Uncomment when using Google Colab\nfrom google.colab import drive\ndrive.mount('/content/drive')\n

In [ ]:
!git clone https://github.com/visheshGupta01/STM-Spectrogram-Transformer-Model-for-Underwater-Acoustic-Target-Recognition.git\n%cd STM-Spectrogram-Transformer-Model-for-Underwater-Acoustic-Target-Recognition\n!pip install -r requirements.txt\nimport sys, os\nsys.path.insert(0, os.path.abspath('src'))\nimport torch\nprint('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 2. Dataset metadata

Create a CSV with `audio_path`, `class_label` (A-E), and `recording_order`. Use `scripts/inspect_excel_metadata.py` to inspect the supplied workbook, then map its actual filename, vessel, and category fields. Do not infer a class from an ambiguous filename.

In [ ]:
# Set these paths to your actual Google Drive locations.\nMETADATA = '/content/drive/MyDrive/ShipsEar/metadata.csv'\nAUDIO_ROOT = '/content/drive/MyDrive/ShipsEar'\nOUTPUT_DATA = 'data/processed'\nassert os.path.exists(METADATA), f'Metadata CSV not found: {METADATA}'

## 3. Create 5-second samples, Dataset A and Dataset B

Dataset A randomly splits clips 70:15:15. Dataset B sorts source recordings by `recording_order`, assigns the earliest 70% of recordings to training, and divides the remainder between validation and test. All clips from one source recording remain in the same Dataset B split.

In [ ]:
from ships_ear.data import prepare_clips\nmanifest = prepare_clips(METADATA, AUDIO_ROOT, OUTPUT_DATA, sample_seconds=5, target_sr=16000, seed=42)\nmanifest.head(), manifest.groupby('split_A').size(), manifest.groupby('split_B').size()

## 4. Extract the three paper features

STFT uses a 254-point FFT and Hann window. FBank uses 128 mel filters with a 10 ms hop. MFCC is computed from the mel representation. Features are cached as NumPy arrays to avoid recomputing them every epoch.

In [ ]:
from ships_ear.features import cache_features\nfeature_manifests = {}\nfor feature in ['stft', 'fbank', 'mfcc']:\n    feature_manifests[feature] = str(cache_features(f'{OUTPUT_DATA}/clips_manifest.csv', OUTPUT_DATA, feature))\nfeature_manifests

## 5. Model definitions

CNN follows the paper's two convolutional layers, Tanh activations, max-pooling, dropout 0.1, and two fully connected layers. ResNet18 is the standard ResNet18 with its first convolution modified for a single-channel spectrogram.

In [ ]:
from ships_ear.models import build_model\ncnn = build_model('cnn')\nresnet = build_model('resnet18')\nprint(cnn)\nprint(resnet)

## 6. Train and evaluate

The paper uses batch size 24, up to 100 epochs, cross-entropy loss, Adam, and learning rate 1e-4 for experiments without pretraining. Set a smaller epoch count for a smoke test, then run the full settings.

In [ ]:
from ships_ear.engine import train_model\nEPOCHS = 100\nBATCH_SIZE = 24\nLEARNING_RATE = 1e-4\nresults = []\nfor dataset in ['A', 'B']:\n    for feature in ['stft', 'fbank', 'mfcc']:\n        for model in ['cnn', 'resnet18']:\n            print(f'\\n=== Dataset {dataset} | {feature} | {model} ===')\n            results.append(train_model(feature_manifests[feature], dataset, feature, model,\n                output_dir='outputs', epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE))

## 7. Compare with the paper

Paper reference values (not results from this notebook):

| Model/feature | Dataset A | Dataset B |
|---|---:|---:|
| ResNet18 + STFT (Table 2) | 76.9% | 60.1% |
| ResNet18 + FBank (Table 2) | 93.8% | 88.2% |
| ResNet18 + MFCC (Table 2) | 93.8% | 85.5% |
| CNN baseline (Table 4) | 84.0% | 39.9% |
| ResNet18 baseline (Table 4) | 93.7% | 88.1% |

The paper's Table 2 specifically compares feature types on ResNet18. Table 4 compares model baselines. Your reproduced metrics are stored under `outputs/`; compare them only after the experiments finish.

In [ ]:
import pandas as pd\nsummary = pd.DataFrame([{k: r[k] for k in ['dataset','feature','model','test_accuracy','best_validation_accuracy','best_epoch','test_samples']} for r in results])\nsummary.sort_values(['dataset','model','feature'])

## 8. Result artifacts and interpretation

Each experiment saves a best-validation checkpoint, metrics JSON, training/validation curves, and a test confusion matrix. Discuss differences from the paper in terms of preprocessing, split reconstruction, metadata chronology, random seed, architecture details that the paper does not fully specify, and hardware/software differences. Do not claim paper-level reproduction unless the dataset mapping and split counts have been verified.